# Entropy Gate Pipeline — Colab runner

Before running anything, set your secrets: click the **key icon** in the left sidebar and add:

| Name | Value |
|---|---|
| `GITHUB_TOKEN` | A [personal access token](https://github.com/settings/tokens) with `repo` scope — needed because the repo is private |
| `APCA_API_KEY_ID` | Your Alpaca key ID |
| `APCA_API_SECRET_KEY` | Your Alpaca secret key |

Toggle **Notebook access** on for each one. Nothing here ever prints these values or writes them anywhere but a local `.env` file — that file never leaves this runtime and is never committed.

**Runtime → Change runtime type → T4 GPU** (or better, if Colab Pro offers it) before running.

In [ ]:
import torch
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU attached — go to Runtime -> Change runtime type -> GPU")

## 1. Mount Drive

Colab's local disk is wiped every time the runtime recycles. `data/` (cached bars) and `outputs/` (screen results, trained metrics) get symlinked into a Drive folder instead, so a day's Alpaca download or a training run survives a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/entropy-gate-pipeline-data'
import os
os.makedirs(f'{DRIVE_DATA_DIR}/data', exist_ok=True)
os.makedirs(f'{DRIVE_DATA_DIR}/outputs', exist_ok=True)
print(f'persistent storage ready at {DRIVE_DATA_DIR}')

## 2. Get the code

Clones fresh each session (the repo is small — this takes a couple seconds). Re-running this cell after you've pushed local changes just pulls the update instead of re-cloning.

In [ ]:
from google.colab import userdata

GITHUB_REPO = 'Gaire-commits/entropy-gate-pipeline'
REPO_DIR = '/content/entropy-gate-pipeline'

token = userdata.get('GITHUB_TOKEN')
clone_url = f'https://{token}@github.com/{GITHUB_REPO}.git'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print('repo already present — pulling latest')
    !cd {REPO_DIR} && git pull
else:
    !git clone {clone_url} {REPO_DIR}

%cd {REPO_DIR}

## 3. Link persistent storage, install dependencies, write `.env`

In [ ]:
for name in ('data', 'outputs'):
    local_path = f'{REPO_DIR}/{name}'
    drive_path = f'{DRIVE_DATA_DIR}/{name}'
    if os.path.islink(local_path) or os.path.exists(local_path):
        if not os.path.islink(local_path):
            raise RuntimeError(f'{local_path} exists as a real directory, not a symlink — remove it before re-running')
    else:
        os.symlink(drive_path, local_path)
print('data/ and outputs/ point at Drive')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
key = userdata.get('APCA_API_KEY_ID')
secret = userdata.get('APCA_API_SECRET_KEY')

with open('.env', 'w') as f:
    f.write(f'APCA_API_KEY_ID={key}\n')
    f.write(f'APCA_API_SECRET_KEY={secret}\n')
    f.write('ALPACA_DATA_FEED=iex\n')

del key, secret  # out of the notebook's own variable table too
print('.env written — keys are not printed and .env is gitignored')

## 4. Sanity check

No credentials needed for this part — confirms the pipeline itself still works in this environment before spending any GPU time.

In [ ]:
!python -m pytest tests/ -q

In [ ]:
!python scripts/smoke_test.py

## 5. Real data: fetch, screen, train

Each writes to `data/`/`outputs/` on Drive, so re-running later skips what's already cached instead of re-downloading.

In [ ]:
!python scripts/fetch_data.py

In [ ]:
!python scripts/run_screen.py

In [ ]:
!python scripts/train.py --arch resnet1d --gate

## 6. Sweep every architecture

Runs the whole ladder back to back — the comparison the thesis actually turns on. Increase `epochs` in `config.yaml` once this runs clean; it's left low here to keep the first pass fast.

In [ ]:
archs_1d = ['logreg', 'cnn1d', 'resnet1d', 'inceptiontime']
for arch in archs_1d:
    print(f'\n{"="*70}\n{arch}\n{"="*70}')
    !python scripts/train.py --arch {arch} --gate

In [ ]:
# resnet2d consumes GAF images, not the 1-d sequence config.yaml sets by default.
# Write a derived config rather than editing config.yaml in place, so the base
# config never silently drifts between reruns.
import yaml
with open('config.yaml') as f:
    gaf_cfg = yaml.safe_load(f)
gaf_cfg['features']['encoding'] = 'gaf'
gaf_cfg['model']['arch'] = 'resnet2d'
with open('config_gaf.yaml', 'w') as f:
    yaml.safe_dump(gaf_cfg, f)

!python scripts/train.py --config config_gaf.yaml --arch resnet2d --gate

## 7. Pull results back down

`outputs/` already lives on Drive via the symlink, so `summary_*.json` and `walkforward_*.csv` for every run above are sitting in **My Drive → entropy-gate-pipeline-data → outputs** — no extra download step needed.

In [ ]:
import json, glob
for path in sorted(glob.glob('outputs/summary_*.json')):
    with open(path) as f:
        s = json.load(f)
    print(f"{s['arch']:15s} gate={str(s['gate']):5s}  acc {s['accuracy_mean']:.3f}  "
          f"net {s['net_bps_mean']:+7.2f}bps  breakeven {s['breakeven_cost_bps']:6.2f}bps  "
          f"profitable {s['folds_profitable_net']}/{s['folds']} folds")